<a href="https://colab.research.google.com/github/NBekhruzbek/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install transformers  bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.1 MB/s eta 0:00:00


In [19]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

In [22]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens map', tokenizer.special_tokens_map)


# ---------------------------------------------------------------
# QUANTIZATION (kvantlash)
# Model og'irliklarini (weights) 16-bit o'rniga 4-bitda saqlaymiz.
# Natija: GPU xotirasi ~3-4 barobar kamayadi (~2.2 GB -> ~0.7 GB),
# aniqlik esa juda oz pasayadi.
# QLoRA fine-tuning uchun asos bo'ladi.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # og'irliklarni 4-bitda yuklash
    bnb_4bit_use_double_quant=True,         # scale konstantalarini ham kvantlash (qo'shimcha xotira tejash)
    bnb_4bit_compute_dtype=torch.bfloat16,  # hisoblashlar bfloat16 da bajariladi (saqlash 4-bit, hisob 16-bit)
    bnb_4bit_quant_type='nf4'               # NF4 (NormalFloat4): normal taqsimotdagi og'irliklar uchun optimal format
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # GPU, CPU
    dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [35]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is."
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)

first_block = model.model.layers[0]
print('frist_block:', first_block)
print('================================')
print(first_block.self_attn)
print('================================')
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is. How does it work and what are its main benefits?
frist_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)


In [36]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272
